# Depression scores and demographic data

Explore the public Depresjon dataset, predict follow-up MADRS scores from baseline scores, and examine affective-type classification from education.

Data: Garcia-Ceja et al. (2018), [Depresjon](https://doi.org/10.5281/zenodo.1219550), licensed [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/). The included `scores.csv` matches the official release. See [README](README.md) for the data dictionary and interpretation limits.


## Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

raw_df = pd.read_csv("scores.csv")
df = raw_df.copy()

df.head()


## Prepare data for exploration

In [ ]:
# Keep observed scores for descriptive analyses.
print("Missing values by column:")
print(raw_df.isna().sum())
df = raw_df.dropna(subset=["afftype"]).copy()
print(f"Participants with recorded affective type: {len(df)}")


## Explore scores and demographics

In [ ]:
# Group Level Observation
sns.barplot(x='afftype', y='madrs1', data=df, errorbar=None)
plt.title('Comparison of Mean MADRS1 Scores between Different Types of Depression')
plt.show()

# Plot a boxplot comparing MADRS1 scores between diff types of depressions
sns.boxplot(x='afftype', y='madrs1', data=df)
plt.title('Comparison of MADRS1 Scores between  Different Types of Depression')
plt.show()

# Comparison between Different Types of Depression
# Plot a boxplot comparing MADRS1 scores between different types of depression
sns.boxplot(x='afftype', y='madrs2', data=df)
plt.title('Comparison of MADRS2 Scores between Different Types of Depression')
plt.show()


# Plot a bar chart comparing mean MADRS2 scores between diff types of depressions
sns.barplot(x='afftype', y='madrs2', data=df, errorbar=None)
plt.title('Comparison of Mean MADRS2 Scores between  Different Types of Depression')
plt.show()


In [ ]:
# Compare MADRS scores based on education level
plt.figure(figsize=(10, 6))
sns.boxplot(x='edu', y='madrs1', data=df)
plt.title('Comparison of MADRS1 Scores based on Education Level')
plt.xlabel('Education Level')
plt.ylabel('MADRS1 Score')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(x='afftype', y='days', data=df, palette='Pastel1')
plt.title('Days of Measurement Comparison between Condition Group Patients')
plt.xlabel('Afftype')
plt.ylabel('Days of Measurement')
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
sns.histplot(df, x='age', hue='afftype', multiple='stack', binwidth=5, palette='viridis')
plt.title('Age Distribution Comparison between Condition Group Patients')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()



In [ ]:

scores_data = pd.read_csv('scores.csv')

# Scatter plot with regression line for MADRS1 and MADRS2 correlation
plt.figure(figsize=(10, 6))
sns.regplot(x='madrs1', y='madrs2', data=scores_data, scatter_kws={'s': 50}, line_kws={'color': 'red'})

plt.title('Correlation between MADRS1 and MADRS2 Scores')
plt.xlabel('MADRS1 Score')
plt.ylabel('MADRS2 Score')
plt.show()


In [ ]:

scores_data = pd.read_csv('scores.csv')

# Countplot for 'afftype' based on employment status ('work')
plt.figure(figsize=(10, 6))
sns.countplot(x='afftype', hue='work', data=scores_data)

plt.title('Distribution of afftype based on Employment Status (work)')
plt.xlabel('afftype')
plt.ylabel('Count')
plt.legend(title='Work Status', loc='upper right')
plt.show()


In [ ]:
#Subject level observation
condition_col = scores_data['number'].head(23)
print(condition_col)

plt.title('Distribution of MADRS1 score for all condition patients')
sns.lineplot(x=condition_col, y="madrs1", data=scores_data) 
plt.xticks(rotation=90, fontsize='10')
plt.show()

In [ ]:
# Countplot for 'afftype' based on marriage status ('marriage')
plt.figure(figsize=(10, 6))
sns.countplot(x='afftype', hue='marriage', data=scores_data)

plt.title('Distribution of afftype based on Marriage Status')
plt.xlabel('afftype')
plt.ylabel('Count')
plt.legend(title='Marriage Status', loc='upper right')
plt.show()

In [ ]:
# Countplot for 'afftype' based on melancholia status ('melanch')
sns.countplot(x='afftype', hue='melanch', data=scores_data)

plt.title('Distribution of afftype based on Melancholia Status')
plt.xlabel('afftype')
plt.ylabel('Count')
plt.legend(title='Melancholia', loc='upper right')
plt.show()

In [ ]:
# Subject Level Observation
# Compare MADRS1 scores across subjects; the x-axis is subject identity, not time.
condition_col = scores_data['number'].head(23)
sns.lineplot(x=condition_col, y='madrs1', hue='afftype', data=scores_data)
plt.title('Individual Subject Level MADRS1 Scores In Accordance to Each Depression Type')
plt.xticks(rotation=90, fontsize=5)
plt.show()


## Linear regression

In [ ]:
# Predict MADRS2 from MADRS1 using observed target values.
# Fit missing-feature imputation only on the training set.
model_data = raw_df[['madrs1', 'madrs2']].copy()
model_data['madrs1'] = pd.to_numeric(model_data['madrs1'], errors='coerce')
model_data['madrs2'] = pd.to_numeric(model_data['madrs2'], errors='coerce')
model_data = model_data.dropna(subset=['madrs2'])
X = model_data[['madrs1']].to_numpy()
y = model_data['madrs2'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred)}")
print(f"Model Coefficients: {model.coef_}")
print(f"Model Intercept: {model.intercept_}")
new_madrs1 = np.array([[20]])
predicted_madrs2 = model.predict(imputer.transform(new_madrs1))
print(f"Predicted MADRS2 for MADRS1 = 20: {predicted_madrs2[0]}")


## Exploratory logistic regression

In [ ]:
# Exploratory logistic regression: afftype from education.
df = raw_df.dropna(subset=['edu', 'afftype']).copy()
if df['edu'].dtype == 'object':
    label_encoder = LabelEncoder()
    df['edu_encoded'] = label_encoder.fit_transform(df['edu'])
    feature_col = 'edu_encoded'
else:
    feature_col = 'edu'

# Selecting the feature and the target variable
X = df[[feature_col]]  # Predictor variable
y = df['afftype']  # Target variable

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initializing the Logistic Regression model
logreg = LogisticRegression(random_state=42)

# Training the model
logreg.fit(X_train, y_train)

# Making predictions on the testing set
y_pred = logreg.predict(X_test)

# Report undefined precision/recall as zero rather than perfect scores.
print(classification_report(y_test, y_pred, zero_division=0))
